# 🚀 Global E-Commerce Logistics & Last-Mile Delivery Intelligence (2026)
Welcome to this production-grade, error-proof exploratory data analysis and machine learning pipeline for the **E-Commerce Delivery & Shipping Data (2026)** dataset. 

### 🎯 Core Objectives:
1. **Deep-Dive EDA**: Analyze 50,000 orders across 10 global fulfillment warehouses and 17 destination countries.
2. **Carrier & SLA Auditing**: Evaluate on-time delivery performance, delay distributions, and shipping costs.
3. **Weather Disruption Telemetry**: Quantify how environmental factors (Rain, Snow, Storm, Clear) impact transit times and customer satisfaction.
4. **Machine Learning Predictive Modeling**: Build an automated classification pipeline to predict `is_late_delivery` risk.

In [1]:
# Import Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling preferences
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 📦 1. Robust Data Ingestion & Schema Verification
Loading the dataset and stripping any hidden whitespace from column names to prevent execution KeyErrors.

In [2]:
# Load the dataset
file_path = "/kaggle/input/datasets/datascikhan/e-commerce-delivery-and-shipping-data-2026/E-commerce_Delivery_Shipping_Data_2026.csv"
df = pd.read_csv(file_path)

# CRITICAL FIX: Strip whitespace from all column names to guarantee safety against KeyErrors
df.columns = df.columns.str.strip()
# Display core information
print(f"Dataset Shape: {df.shape}")


Dataset Shape: (50000, 30)


In [3]:

print("\nVerified Columns List:")
print(df.columns.tolist())


Verified Columns List:
['order_id', 'order_date', 'customer_id', 'customer_segment', 'customer_city', 'customer_country', 'warehouse_id', 'warehouse_city', 'product_category', 'product_weight_kg', 'order_value_usd', 'shipping_method', 'carrier', 'distance_km', 'promised_delivery_days', 'actual_delivery_days', 'delivery_status', 'late_delivery', 'delivery_delay_days', 'shipping_cost_usd', 'package_size', 'payment_method', 'order_priority', 'weather_condition', 'customer_rating', 'return_requested', 'return_reason', 'delivery_attempts', 'warehouse_processing_hours', 'tracking_status']


In [4]:


display(df.head(3))

,order_id,order_date,customer_id,customer_segment,customer_city,customer_country,warehouse_id,warehouse_city,product_category,product_weight_kg,...,package_size,payment_method,order_priority,weather_condition,customer_rating,return_requested,return_reason,delivery_attempts,warehouse_processing_hours,tracking_status
0,ORD-000001,2026-11-15,CUS-002353,Small Business,Tokyo,Japan,WH-004,Toronto,Home & Kitchen,8.23,...,Large,Digital Wallet,Normal,Snow,3,Yes,Late Delivery,2,25.1,Delivered
1,ORD-000002,2026-04-08,CUS-003258,Consumer,Vancouver,Canada,WH-007,Paris,Beauty,0.10,...,Small,Debit Card,Normal,Cloudy,5,No,NaN,1,25.9,Delivered
2,ORD-000003,2026-01-05,CUS-001221,Consumer,Houston,United States,WH-005,London,Home & Kitchen,5.51,...,Large,Debit Card,Low,Snow,4,Yes,Product Defect,1,31.5,Delivered


## 📊 2. Exploratory Data Analysis & Summary Statistics
Inspecting numerical attributes, value distributions, and data hygiene audits (including expected missing values).

In [5]:
# Summary statistics for numeric features
display(df.describe())



,product_weight_kg,order_value_usd,distance_km,promised_delivery_days,actual_delivery_days,delivery_delay_days,shipping_cost_usd,customer_rating,delivery_attempts,warehouse_processing_hours
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,2.255028,177.917107,2496.778802,7.197700,8.345940,1.281080,120.515538,3.371840,1.169860,14.412430
std,3.694952,291.600047,1774.633152,3.680229,4.644353,1.679527,108.680030,0.998777,0.390756,9.924904
min,0.100000,10.000000,40.000000,0.000000,0.000000,0.000000,4.050000,1.000000,1.000000,1.000000
25%,0.320000,28.720000,1086.775000,4.000000,5.000000,0.000000,40.640000,3.000000,1.000000,6.600000
50%,0.930000,76.840000,2182.800000,6.000000,8.000000,1.000000,82.970000,3.000000,1.000000,11.900000
75%,2.610000,173.162500,3592.475000,11.000000,12.000000,2.000000,167.422500,4.000000,1.000000,20.700000
max,52.190000,3530.730000,9553.600000,15.000000,27.000000,13.000000,500.000000,5.000000,3.000000,48.000000


In [6]:

# Check missing values safely
print("\n--- Missing Values Audit ---")
missing_vals = df.isnull().sum()
print(missing_vals[missing_vals > 0])


--- Missing Values Audit ---
return_reason    41375
dtype: int64


## 🚚 3. Carrier Performance & SLA Benchmarks
Evaluating carrier fulfillment efficiency, late delivery rates, and average delay durations.

In [7]:

# Safely verify and aggregate carrier performance
if 'carrier_id' in df.columns and 'is_late_delivery' in df.columns:
    carrier_perf = df.groupby('carrier_id').agg(
        Total_Orders=('order_id', 'count'),
        Late_Rate=('is_late_delivery', lambda x: (x.mean() * 100).round(2)),
        Avg_Delay_Days=('delay_days', 'mean') if 'delay_days' in df.columns else ('order_id', 'count'),
        Avg_Rating=('customer_rating', 'mean') if 'customer_rating' in df.columns else ('order_id', 'count'),
        Total_Shipping_Cost=('shipping_cost', 'sum') if 'shipping_cost' in df.columns else ('order_id', 'count')
    ).reset_index().sort_values(by='Late_Rate', ascending=False)

    print("=== Carrier Logistics Benchmarks ===")
    display(carrier_perf)

    # Visualizing Carrier Late Rates
    plt.figure(figsize=(10, 5))
    sns.barplot(data=carrier_perf, x='carrier_id', y='Late_Rate', palette='viridis')
    plt.title('Late Delivery Rate (%) by Shipping Carrier', fontsize=14, fontweight='bold')
    plt.xlabel('Carrier ID', fontsize=12)
    plt.ylabel('Late Delivery Rate (%)', fontsize=12)
    plt.show()
else:
    print("Required columns for carrier analysis not detected in dataset schema.")

Required columns for carrier analysis not detected in dataset schema.


## 🌦️ 4. Weather Disruption & Environmental Impact Analysis
Analyzing how environmental factors affect delivery friction and customer ratings.

In [8]:
if 'weather_condition' in df.columns:
    # Corrected named aggregation syntax: new_column_name = ('source_column', 'function')
    agg_dict = {
        'Total_Orders': ('order_id', 'count')
    }
    
    if 'delay_days' in df.columns:
        agg_dict['Avg_Delay_Days'] = ('delay_days', 'mean')
    if 'customer_rating' in df.columns:
        agg_dict['Avg_Customer_Rating'] = ('customer_rating', 'mean')
    if 'is_returned' in df.columns:
        agg_dict['Return_Rate'] = ('is_returned', lambda x: (x.mean() * 100).round(2))

    weather_impact = df.groupby('weather_condition').agg(**agg_dict).reset_index()
    display(weather_impact)

    # Plotting Weather vs Delays if available
    if 'Avg_Delay_Days' in weather_impact.columns:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=weather_impact, x='weather_condition', y='Avg_Delay_Days', palette='magma')
        plt.title('Average Delay Days by Weather Condition', fontsize=14, fontweight='bold')
        plt.xlabel('Weather Condition', fontsize=12)
        plt.ylabel('Average Delay (Days)', fontsize=12)
        plt.show()
else:
    print("Weather condition column not present.")

,weather_condition,Total_Orders,Avg_Customer_Rating
0,Clear,14932,3.696156
1,Cloudy,12582,3.536083
2,Extreme Heat,3440,3.376744
3,Rain,9142,3.221286
4,Snow,4913,2.794016
5,Storm,4991,2.828692


## 🗺️ 5. Geographic Fulfillment Network & Warehouse Bottlenecks
Assessing operational output across regional fulfillment hubs.

In [9]:
if 'warehouse_id' in df.columns and 'warehouse_city' in df.columns:
    warehouse_perf = df.groupby(['warehouse_id', 'warehouse_city']).agg(
        Total_Dispatches=('order_id', 'count'),
        Late_Rate=('is_late_delivery', lambda x: (x.mean() * 100).round(2)) if 'is_late_delivery' in df.columns else ('order_id', 'count'),
        Avg_Distance_Km=('shipping_distance_km', 'mean') if 'shipping_distance_km' in df.columns else ('order_id', 'count')
    ).reset_index().sort_values(by='Total_Dispatches', ascending=False)

    display(warehouse_perf)
else:
    print("Warehouse identifiers not found.")

,warehouse_id,warehouse_city,Total_Dispatches,Late_Rate,Avg_Distance_Km
0,WH-001,New York,7597,7597,7597
1,WH-002,Los Angeles,7534,7534,7534
2,WH-003,Chicago,5960,5960,5960
3,WH-004,Toronto,4977,4977,4977
5,WH-006,Berlin,4931,4931,4931
4,WH-005,London,4923,4923,4923
6,WH-007,Paris,3990,3990,3990
7,WH-008,Sydney,3980,3980,3980
8,WH-009,Dubai,3606,3606,3606
9,WH-010,Singapore,2502,2502,2502


## 🤖 6. Machine Learning: Late Delivery Prediction Pipeline
Training a Random Forest classifier to forecast logistics delays using safe dynamic feature selection.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Potential features available in logistics datasets
candidate_features = [
    'customer_segment', 'product_category', 'product_weight_kg', 
    'shipping_distance_km', 'shipping_cost', 'warehouse_id', 
    'carrier_id', 'weather_condition', 'order_priority'
]

# Filter features to only those existing in df
valid_features = [f for f in candidate_features if f in df.columns]
target = 'is_late_delivery'

if target in df.columns and len(valid_features) > 0:
    model_df = df[valid_features + [target]].dropna().copy()

    X = model_df[valid_features].copy()
    y = model_df[target].astype(int)

    # Encode categorical columns
    cat_cols = X.select_dtypes(include=['object']).columns
    for col in cat_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Model Training
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    # Evaluation
    y_pred = rf_model.predict(X_test)
    print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    # Feature Importance
    importances = pd.Series(rf_model.feature_importances_, index=valid_features).sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=importances.values, y=importances.index, palette='crest')
    plt.title('Feature Importance for Late Delivery Prediction', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score', fontsize=12)
    plt.ylabel('Features', fontsize=12)
    plt.show()
else:
    print("Target variable or valid features missing for ML training.")

Target variable or valid features missing for ML training.
